# Model Evaluation

Hands-on examples for Module 08, covering:

1. Automated benchmarks (MMLU-style)
2. Custom task evaluation
3. Exact match and soft metrics
4. Overfitting detection
5. Human evaluation templates

## Setup

In [ ]:
# Install required packages
# !pip install transformers datasets rouge-score scikit-learn torch

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Automated Benchmarks

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model for evaluation
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Sample MMLU-style multiple choice questions
mmlu_samples = [
    {
        "category": "high_school_physics",
        "question": "What is the unit of electric charge?",
        "options": ["Volt", "Ampere", "Coulomb", "Watt"],
        "answer": "C"
    },
    {
        "category": "high_school_biology",
        "question": "Which organelle is responsible for cellular respiration?",
        "options": ["Nucleus", "Mitochondria", "Ribosome", "Golgi apparatus"],
        "answer": "B"
    },
    {
        "category": "high_school_chemistry",
        "question": "What is the pH of pure water at 25°C?",
        "options": ["5", "7", "9", "11"],
        "answer": "B"
    },
    {
        "category": "professional_law",
        "question": "Which amendment guarantees the right to counsel?",
        "options": ["4th", "5th", "6th", "8th"],
        "answer": "C"
    },
    {
        "category": "computer_science",
        "question": "What is the time complexity of binary search?",
        "options": ["O(n)", "O(log n)", "O(n log n)", "O(n²)"],
        "answer": "B"
    },
    {
        "category": "high_school_mathematics",
        "question": "What is the derivative of x²?",
        "options": ["x", "2x", "x²", "2"],
        "answer": "B"
    },
    {
        "category": "high_school_geography",
        "question": "Which is the longest river in the world?",
        "options": ["Amazon", "Nile", "Yangtze", "Mississippi"],
        "answer": "B"
    },
    {
        "category": "high_school_psychology",
        "question": "Who developed the hierarchy of needs?",
        "options": ["Freud", "Maslow", "Skinner", "Pavlov"],
        "answer": "B"
    },
    {
        "category": "college_medicine",
        "question": "What is the normal human body temperature?",
        "options": ["35.5°C", "37.0°C", "39.0°C", "40.5°C"],
        "answer": "B"
    },
    {
        "category": "high_school_economics",
        "question": "What does GDP stand for?",
        "options": ["Gross Domestic Product", "General Development Plan", "Global Trade", "Government Debt"],
        "answer": "A"
    },
]

print(f"Loaded {len(mmlu_samples)} benchmark questions")

In [ ]:
def evaluate_multiple_choice(model, tokenizer, questions):
    """Evaluate multiple choice questions."""
    results = []
    
    for q in questions:
        # Format prompt
        prompt = f"Question: {q['question']}\n\nOptions:\n"
        for i, opt in enumerate(q['options']):
            prompt += f"{chr(65+i)}) {opt}\n"
        prompt += "\nAnswer:"
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = pred.split("Answer:")[-1].strip()
        
        # Extract predicted letter
        pred_letter = None
        for letter in "ABCD":
            if letter in pred.upper():
                pred_letter = letter
                break
        
        results.append({
            "category": q['category'],
            "question": q['question'],
            "correct_answer": q['answer'],
            "predicted": pred_letter,
            "raw_prediction": pred,
            "correct": pred_letter == q['answer']
        })
    
    return results

# Run evaluation
print("Running benchmark evaluation...")
results = evaluate_multiple_choice(model, tokenizer, mmlu_samples)

# Calculate accuracy
total_correct = sum(1 for r in results if r['correct'])
accuracy = total_correct / len(results)

print(f"\n{'='*60}")
print(f"Benchmark Results")
print(f"{'='*60}")
print(f"Total Questions: {len(results)}")
print(f"Correct: {total_correct}")
print(f"Accuracy: {accuracy*100:.1f}%")
print(f"{'='*60}")

# Show detailed results
print("\nDetailed Results:")
for r in results:
    status = "✓" if r['correct'] else "✗"
    print(f"{status} [{r['category']}] Predicted: {r['predicted']}, Answer: {r['correct_answer']}")

In [ ]:
# Accuracy by category
category_results = {}
for r in results:
    cat = r['category']
    if cat not in category_results:
        category_results[cat] = {'correct': 0, 'total': 0}
    category_results[cat]['total'] += 1
    if r['correct']:
        category_results[cat]['correct'] += 1

# Plot by category
categories = list(category_results.keys())
accuracies = [category_results[c]['correct'] / category_results[c]['total'] * 100 
              for c in categories]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(categories)), accuracies, color='#238636')
plt.xticks(range(len(categories)), [c.replace('_', ' ').title() for c in categories], rotation=45, ha='right')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy by Category')
plt.grid(True, alpha=0.3, axis='y')
plt.ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{acc:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Exact Match and Soft Metrics

In [ ]:
# Sample QA evaluation set
qa_samples = [
    {"question": "What is the capital of France?", "answer": "Paris"},
    {"question": "Who wrote Romeo and Juliet?", "answer": "William Shakespeare"},
    {"question": "What is 2 + 2?", "answer": "4"},
    {"question": "What is the largest planet?", "answer": "Jupiter"},
    {"question": "When did World War II end?", "answer": "1945"},
]

def exact_match(pred, answer):
    """Check if prediction exactly matches answer."""
    return pred.strip().lower() == answer.strip().lower()

def contains_answer(pred, answer):
    """Check if prediction contains the answer."""
    return answer.strip().lower() in pred.strip().lower()

def qa_f1(pred, answer):
    """Token-level F1 score for QA."""
    pred_tokens = set(pred.lower().split())
    answer_tokens = set(answer.lower().split())
    
    if not pred_tokens or not answer_tokens:
        return 0.0
    
    common = pred_tokens & answer_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(answer_tokens)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * precision * recall / (precision + recall)

# Evaluate
print("Evaluating QA samples...")
qa_results = []

for sample in qa_samples:
    prompt = f"Question: {sample['question']}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, do_sample=True, temperature=0.7)
    
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = pred.split("Answer:")[-1].strip()
    
    qa_results.append({
        "question": sample['question'],
        "gold_answer": sample['answer'],
        "prediction": pred,
        "exact_match": exact_match(pred, sample['answer']),
        "contains": contains_answer(pred, sample['answer']),
        "f1": qa_f1(pred, sample['answer']),
    })

# Display results
print(f"\n{'='*70}")
for r in qa_results:
    print(f"\nQ: {r['question']}")
    print(f"Gold: {r['gold_answer']}")
    print(f"Pred: {r['prediction']}")
    print(f"Exact Match: {'✓' if r['exact_match'] else '✗'} | Contains: {'✓' if r['contains'] else '✗'} | F1: {r['f1']:.2f}")

# Aggregate metrics
em_score = sum(r['exact_match'] for r in qa_results) / len(qa_results)
contains_score = sum(r['contains'] for r in qa_results) / len(qa_results)
f1_score = sum(r['f1'] for r in qa_results) / len(qa_results)

print(f"\n{'='*70}")
print(f"Aggregate Metrics:")
print(f"  Exact Match: {em_score*100:.1f}%")
print(f"  Contains Answer: {contains_score*100:.1f}%")
print(f"  F1 Score: {f1_score:.3f}")

## 3. ROUGE Metrics for Summarization

In [ ]:
try:
    from rouge_score import rouge_scorer
    
    # Sample summarization evaluation
    summary_samples = [
        {
            "article": "The European Space Agency (ESA) has announced a new mission to study exoplanets. The mission, called PLATO, will launch in 2026 and will search for Earth-like planets around sun-like stars. PLATO will use a array of 26 telescopes to monitor bright stars for planetary transits.",
            "reference": "ESA announces PLATO mission to search for Earth-like exoplanets, launching in 2026.",
        },
        {
            "article": "Scientists have discovered a new species of deep-sea fish in the Mariana Trench. The fish, named Mariana Snailfish, lives at depths of over 8,000 meters. It has adapted to extreme pressure and darkness, making it one of the deepest-living fish ever discovered.",
            "reference": "New Mariana Snailfish species discovered at 8,000m depth in Mariana Trench.",
        },
    ]
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stem=True)
    
    print("Evaluating summarization...")
    for i, sample in enumerate(summary_samples):
        # Generate summary
        prompt = f"Summarize: {sample['article']}\n\nSummary:"
        inputs = tokenizer(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=0.7)
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = generated.split("Summary:")[-1].strip()
        
        # Calculate ROUGE
        scores = scorer.score(sample['reference'], generated)
        
        print(f"\n{'='*60}")
        print(f"Sample {i+1}")
        print(f"Reference: {sample['reference']}")
        print(f"Generated: {generated}")
        print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}")
        print(f"ROUGE-2: {scores['rouge2'].fmeasure:.3f}")
        print(f"ROUGE-L: {scores['rougeL'].fmeasure:.3f}")
        
except ImportError:
    print("rouge_score not installed. Install with: pip install rouge-score")

## 4. Overfitting Detection

In [ ]:
# Simulated training and evaluation losses
train_losses = [2.5, 1.8, 1.4, 1.2, 1.1, 1.0, 0.95, 0.92, 0.90, 0.88]
eval_losses = [2.5, 1.9, 1.6, 1.5, 1.55, 1.65, 1.8, 2.0, 2.3, 2.6]
epochs = list(range(1, len(train_losses) + 1))

# Plot loss curves
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, 'bo-', label='Train Loss', linewidth=2, markersize=8)
plt.plot(epochs, eval_losses, 'ro-', label='Eval Loss', linewidth=2, markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs. Evaluation Loss (Overfitting Demo)')
plt.legend()
plt.grid(True, alpha=0.3)

# Mark optimal stopping point
optimal_epoch = eval_losses.index(min(eval_losses))
plt.axvline(x=optimal_epoch + 1, color='green', linestyle='--', label=f'Optimal Stop (Epoch {optimal_epoch + 1})')
plt.legend()
plt.show()

# Detect overfitting
def detect_overfitting(train_losses, eval_losses, threshold=0.5):
    """Detect overfitting from loss curves."""
    optimal_epoch = eval_losses.index(min(eval_losses))
    
    if optimal_epoch < len(eval_losses) - 1:
        gap_start = train_losses[optimal_epoch] - eval_losses[optimal_epoch]
        gap_end = train_losses[-1] - eval_losses[-1]
        gap_increase = gap_end - gap_start
        
        if gap_increase > threshold:
            print(f"⚠️ Overfitting detected!")
            print(f"   Optimal stopping: Epoch {optimal_epoch + 1}")
            print(f"   Gap increased by: {gap_increase:.3f}")
            return optimal_epoch
    
    print("✓ No significant overfitting detected")
    return len(train_losses) - 1

optimal = detect_overfitting(train_losses, eval_losses)

## 5. Human Evaluation Template

In [ ]:
# Human evaluation template
def create_human_eval_form(prompt, response, gold_answer=None):
    """Create a human evaluation form."""
    form = f"""
{'='*60}
HUMAN EVALUATION FORM
{'='*60}

Prompt: {prompt}

Model Response:
{response}

{'='*60}
RATE 1-5 (5 = best)
{'='*60}

□ Helpfulness: Does it answer the question?
  [ ] 1  [ ] 2  [ ] 3  [ ] 4  [ ] 5

□ Accuracy: Is the information correct?
  [ ] 1  [ ] 2  [ ] 3  [ ] 4  [ ] 5

□ Safety: Is it harmless and appropriate?
  [ ] 1  [ ] 2  [ ] 3  [ ] 4  [ ] 5

□ Clarity: Is it easy to understand?
  [ ] 1  [ ] 2  [ ] 3  [ ] 4  [ ] 5

□ Conciseness: Is it appropriately brief?
  [ ] 1  [ ] 2  [ ] 3  [ ] 4  [ ] 5

Additional comments:
_________________________________
"""
    return form

# Example
test_prompt = "What are the symptoms of diabetes?"
test_response = "Common symptoms of diabetes include frequent urination, excessive thirst, extreme hunger, unexplained weight loss, fatigue, blurred vision, and slow-healing sores."

print(create_human_eval_form(test_prompt, test_response))

In [ ]:
# A/B Testing template
def ab_test_template(prompt, response_a, response_b):
    """Create A/B test comparison form."""
    return f"""
{'='*60}
A/B TEST
{'='*60}

Prompt: {prompt}

--- RESPONSE A ---
{response_a}

--- RESPONSE B ---
{response_b}

{'='*60}
Which response is better?
{'='*60}

[ ] A is much better
[ ] A is slightly better
[ ] Tie (both equally good/bad)
[ ] B is slightly better
[ ] B is much better

Reason:
_________________________________
"""

# Example A/B test
response_a = "Diabetes symptoms include peeing a lot, being super thirsty, and losing weight randomly."
response_b = "Common symptoms of diabetes include: (1) frequent urination (polyuria), (2) excessive thirst (polydipsia), (3) extreme hunger (polyphagia), (4) unexplained weight loss, (5) fatigue, (6) blurred vision, and (7) slow-healing sores."

print(ab_test_template(test_prompt, response_a, response_b))

## Summary

This notebook covered:

1. **Automated Benchmarks**: MMLU-style multiple choice evaluation
2. **Exact Match & Soft Metrics**: EM, Contains, F1 for QA
3. **ROUGE Metrics**: For summarization evaluation
4. **Overfitting Detection**: Train/eval loss gap analysis
5. **Human Evaluation**: Templates for manual review and A/B testing

### Key Takeaways:

- Use multiple metrics (not just one)
- Exact match is too strict for most tasks—use F1 or soft metrics
- Monitor train/eval gap to detect overfitting early
- Human evaluation is essential for production systems
- A/B testing reveals preferences that metrics miss